All percentages are rounded to two decimal place unless otherwise stated

In [1]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_selection import VarianceThreshold
from scipy.spatial.distance import pdist
import joblib
import os

In [2]:
np.random.seed(42)

# SETUP

In [3]:
# Load data
df = pl.read_parquet('../data/raw/ucl_raw_aggregated.parquet')
print(f"Initial shape: {df.shape}")
print(f"Initial columns: {df.columns}")

Initial shape: (608, 31)
Initial columns: ['name', 'title', 'email', 'phone', 'department', 'website', 'research_interests', 'bio', 'profile_url', 'university', 'teaching_interests', 'teaching_module_count', 'teaching_modules', 'profile_id', '_fetch_timestamp', '_source', 'publication_count', 'journal_article_count', 'preprint_count', 'recent_publications', 'first_publication_year', 'latest_publication_year', 'has_doi_count', 'total_citations', 'avg_citations_per_publication', 'max_citations', 'avg_coauthors', 'max_coauthors', 'solo_publications', 'h_index', 'i10_index']


In [4]:
df = df.with_row_index(name='_index')

df = df.drop(['phone', 'department', 'university', 'profile_url', '_fetch_timestamp', '_source'])

print(f"After cleanup: {df.shape}")

After cleanup: (608, 26)


## Section 1: Initial Cleanup

All data quality assessments referenced in this section were confirmed during exploratory data analysis.

The initial preprocessing removes non-informative metadata and zero-variance columns to focus on features relevant to academic clustering. A row index is added to preserve observation identity through subsequent dimensionality reduction and clustering operations. This index enables linkage to identity information stored separately, avoiding the need to carry identifier columns through numerical transformations where they serve no analytical purpose.

Six columns are dropped from the dataset. The `phone` field is always empty in the source data. Both `department` and `university` are single-value columns containing 'Computer Science' and 'UCL' respectively, contributing zero variance to any analysis. The columns `profile_url`, `_fetch_timestamp`, and `_source` represent metadata for data provenance rather than analytical features.

The row index `_index` is assigned sequentially from 0 to 607 and is maintained through all subsequent filtering and transformation operations. Polars preserves row order through standard operations, ensuring that index position remains a stable reference throughout the analysis pipeline. The identity columns `name`, `email`, and `profile_id` are retained in the dataset temporarily and will be separated into a dedicated mapping file after feature engineering is complete, enabling post-clustering interpretation without maintaining these fields in the feature matrix.

No missing values are present in the retained text fields (`research_interests`, `bio`, `teaching_interests`, `website`), eliminating the need for imputation at this stage.

In [5]:
# Binary indicators for profile field completion
df = df.with_columns([
    (pl.col('research_interests') != '').cast(pl.Int8).alias('has_research_interests'),
    (pl.col('bio') != '').cast(pl.Int8).alias('has_bio'),
    (pl.col('teaching_interests') != '').cast(pl.Int8).alias('has_teaching_info'),
    (pl.col('website') != '').cast(pl.Int8).alias('has_website'),
])

print(f"Shape: {df.shape}")

Shape: (608, 30)


## Section 2: Profile Completeness Features

Profile completeness indicates engagement level and professional visibility within the academic community. Binary presence flags capture whether researchers choose to populate optional profile fields, providing insight into their approach to professional self-presentation without making qualitative judgements about content.

Four binary features are created to indicate field completion. The feature `has_research_interests` captures whether the research interests section is populated. The feature `has_bio` indicates whether a biography section is present. The feature `has_teaching_info` identifies whether teaching interests are documented, with approximately 13% of profiles completing this field. Despite the low fill rate, this feature is retained as it may identify teaching-focused academics who prioritise pedagogical engagement. The feature `has_website` indicates whether a personal website is listed in the profile.

Binary flags are preferred over alternative encoding schemes for several reasons. Text length metrics were rejected as verbosity does not correlate with profile quality, and concise profiles may be equally valuable. An aggregate completeness score was rejected as it would create multicollinearity by linearly combining individual flags that clustering algorithms can more effectively evaluate independently. This approach allows algorithms to discover patterns from independent features rather than imposing assumptions about how completeness dimensions should be weighted.

In [6]:
df = df.with_columns(
    (pl.col('research_interests') + ' ' +
     pl.col('bio') + ' ' +
     pl.col('teaching_interests')).str.to_lowercase().alias('_combined_text')
)

# Domain keyword patterns based on arXiv CS categories
# Source: https://arxiv.org/archive/cs (accessed Feb 2026)
domain_patterns = {
    'is_ml_ai_focus': r'machine learning|deep learning|neural network|neural net|'
                      r'reinforcement learning|supervised learning|unsupervised learning|'
                      r'artificial intelligence|\bai\b|intelligent system',

    'is_vision_focus': r'computer vision|\bcv\b|image processing|visual recognition|'
                       r'object detection|image analysis|pattern recognition|scene understanding',

    'is_nlp_focus': r'natural language processing|\bnlp\b|text mining|language model|'
                    r'computational linguistics|speech processing|text analysis',

    'is_security_focus': r'\bsecurity\b|cryptography|privacy|authentication|cybersecurity|'
                         r'information security|encryption|cyber attack',

    'is_systems_focus': r'distributed system|parallel computing|database|cloud computing|'
                        r'computer architecture|operating system|\bos\b|cluster computing',

    'is_theory_focus': r'\balgorithm\b|algorithmic|complexity|computational complexity|'
                       r'theoretical computer science|graph theory|optimization|combinatorial',

    'is_hci_focus': r'human computer interaction|\bhci\b|user interface|\bui\b|'
                    r'interaction design|usability|user experience|\bux\b',

    'is_robotics_focus': r'\brobot\b|robotic|autonomous|motion planning|control system',

    'is_data_science_focus': r'data science|data analysis|big data|data mining|'
                             r'analytics|statistical analysis',

    'is_graphics_focus': r'computer graphics|rendering|visualization|visualisation|'
                         r'3d modeling|3d modelling|visual computing',

    'is_bioinformatics_focus': r'bioinformatics|computational biology|genomics|biomedical|'
                               r'medical imaging|health informatics',

    'is_quantum_focus': r'quantum computing|quantum algorithm|quantum machine|'
                        r'\bqnlp\b|quantum information',

    'is_networks_focus': r'\bnetwork\b|networking|wireless|communication protocol|'
                         r'internet architecture|network protocol',

    'is_software_eng_focus': r'software engineering|software development|software design|'
                             r'programming|code quality|software testing',
}

for feature_name, pattern in domain_patterns.items():
    df = df.with_columns(
        pl.col('_combined_text')
        .str.contains(pattern)
        .fill_null(False)
        .cast(pl.Int8)
        .alias(feature_name)
    )

df = df.drop(['_combined_text', 'research_interests', 'bio', 'teaching_interests', 'website'])

print(f"After Changes: {df.shape}")

After Changes: (608, 40)


## Section 3: Research Domain Classification

Research domain flags capture academic specialisation patterns through binary indicators, enabling multi-label classification that reflects the interdisciplinary nature of contemporary computer science research. The methodology draws on standardised taxonomies whilst accommodating emergent research areas not captured in formal classification schemes.

Keywords are primarily sourced from the arXiv Computer Science subject classifications (https://arxiv.org/archive/cs, accessed February 2026), a peer-validated taxonomy maintained by Cornell University and used across the research community. Eleven categories are mapped directly from arXiv classifications. Two custom categories, Data Science and Bioinformatics, address interdisciplinary areas not explicitly represented in the arXiv taxonomy. These custom categories were identified manually
from the text corpus.

Text from three profile fields is combined for domain detection. The `research_interests` field provides primary research focus declarations, whilst `bio` captures career history and broader research context. The `teaching_interests` field contributes teaching-related research areas. These fields are concatenated and converted to lowercase to create a unified text representation for each profile.

Base arXiv keywords are expanded with synonyms and common abbreviations to improve recall whilst maintaining precision. For example, the computer vision category (cs.CV) includes terms such as "computer vision", "image processing", "visual recognition", and "object detection". Word boundaries are enforced for ambiguous abbreviations to prevent false matches.

The Machine Learning (cs.LG) and Artificial Intelligence (cs.AI) categories are merged into a single feature `is_ml_ai_focus`. This decision reflects the interchangeable use of these terms in contemporary computer science research, where distinctions have become increasingly blurred. The co-occurrence data from exploratory analysis shows ML/AI-related research appearing alongside multiple other domains, confirming its role as a broad foundational area rather than a narrow specialism.

Domains with low coverage are retained where they represent legitimate niche specialisations. Quantum computing, with 2.2% coverage, exemplifies genuine rarity in the field rather than poor keyword selection or data quality issues. Similarly, custom categories are included when term frequency analysis demonstrates their prevalence. Data Science, spanning machine learning, databases, and statistical analysis, captures an interdisciplinary area with 13.8% coverage. Bioinformatics, at the intersection of computer science and biological research, represents 12.0% of profiles and reflects growing computational biology research within computer science departments.

The classification supports multi-label assignment, allowing profiles to match multiple domains simultaneously. This design acknowledges interdisciplinary research where academics work at domain intersections, such as applying machine learning techniques to medical imaging problems. Case-insensitive regex matching is applied to the combined text, with matches encoded as binary indicators (0/1).

| Domain Feature | Representative Keywords |
|---|---|
| is_ml_ai_focus | machine learning, deep learning, neural network, artificial intelligence |
| is_software_eng_focus | software engineering, software development, programming |
| is_security_focus | security, cryptography, privacy, cybersecurity |
| is_robotics_focus | robot, robotic, autonomous, motion planning |
| is_data_science_focus | data science, data analysis, big data, analytics |
| is_theory_focus | algorithm, complexity, theoretical computer science |
| is_hci_focus | human computer interaction, user interface, usability |
| is_bioinformatics_focus | bioinformatics, computational biology, biomedical |
| is_vision_focus | computer vision, image processing, pattern recognition |
| is_networks_focus | network, networking, wireless, internet architecture |
| is_graphics_focus | computer graphics, rendering, visualization |
| is_nlp_focus | natural language processing, text mining, language model |
| is_systems_focus | distributed system, database, cloud computing |
| is_quantum_focus | quantum computing, quantum algorithm |

Following feature extraction, the original text columns (`research_interests`, `bio`, `teaching_interests`, `website`) are dropped from the dataset. These fields cannot be directly used in machine learning models without natural language processing techniques or embedding generation, both of which are beyond the scope of this analysis. The binary domain flags capture the relevant signal for clustering purposes whilst reducing memory overhead. The temporary combined text column used during pattern matching is similarly removed as it serves no purpose in subsequent analysis stages.

In [7]:
df = df.with_columns(
    pl.col('title')
    .str.replace('Dept of Computer Science', '')
    .str.strip_chars()
    .alias('title_clean')
)

df = df.with_columns(
    pl.when(pl.col('title_clean') == '')
    .then(pl.lit('Unknown'))
    .otherwise(pl.col('title_clean'))
    .alias('title_clean')
)


def get_seniority_score(title_str):
    """
    Extract seniority level from academic titles (0-4).
    Based on UK academic hierarchy.

    Tier 0: Non-academic, administrative, or unknown (filtered)
    Tier 1: Students and research/teaching assistants
    Tier 2: Lecturers, research fellows, postdocs
    Tier 3: Senior lecturers, readers, associate professors
    Tier 4: Professors and chairs
    """
    if title_str is None or title_str == '':
        return 0

    t = str(title_str).lower()

    if 'professor' in t:
        if 'associate' not in t and 'assistant' not in t:
            return 4

    if 'chair of' in t or 'chair in' in t:
        return 4

    if any(x in t for x in [
        'senior lecturer',
        'reader',
        'associate professor',
        'senior research fellow',
        'principal research fellow',
        'principal lecturer'
    ]):
        return 3

    if 'professorial' in t and 'research associate' in t:
        return 3

    if any(x in t for x in [
        'assistant professor',
        'lecturer',
        'research fellow',
        'teaching fellow',
        'research associate',
        'postdoc'
    ]):
        if 'senior' not in t and 'principal' not in t and 'professorial' not in t:
            return 2

    if any(x in t for x in [
        'phd',
        'student',
        'pgta',
        'teaching assistant',
        'research assistant'
    ]):
        return 1

    return 0


df = df.with_columns(
    pl.col('title_clean')
    .map_elements(get_seniority_score, return_dtype=pl.Int8)
    .alias('seniority_level')
)

df = df.filter(pl.col('seniority_level') != 0)

df = df.drop(['title_clean', 'title'])

print(f"After title filtering: {df.shape}")

After title filtering: (503, 40)


## Section 4: Academic Seniority Classification

Academic title captures career stage, seniority, and institutional standing within the university hierarchy. Encoding this hierarchical information as an ordinal variable enables clustering algorithms to identify patterns related to career progression and academic status whilst preserving the natural ordering of academic ranks.

The title field is first cleaned by removing the departmental prefix "Dept of Computer Science" and trimming whitespace. Empty titles are coded as "Unknown" to enable subsequent filtering. Titles are then classified into a five-tier hierarchy (0-4) based on UK academic rank structures as documented in Academic ranks in the United Kingdom (n.d., Wikipedia, retrieved February 7, 2026, from https://en.wikipedia.org/wiki/Academic_ranks_in_the_United_Kingdom). This standardised hierarchy is used across UK universities, with UCL-specific structures aligning with the broader national system.

The classification scheme assigns Tier 4 to professors and chairs, representing the most senior academic positions. This includes any title containing "Professor" excluding Associate Professor and Assistant Professor, as well as titled chairs indicated by "Chair of" or "Chair in" patterns. Emeritus professors are retained at this level as they maintain academic standing. Tier 3 captures senior academic staff including Senior Lecturers, Readers, Associate Professors, and Senior Research Fellows. The edge case of "Professorial Research Associate" is classified at Tier 3 due to the senior modifier. Tier 2 encompasses early and mid-career academics such as Lecturers, Assistant Professors, Research Fellows, Teaching Fellows, Research Associates, and Postdocs, excluding those with senior or principal modifiers which elevate them to Tier 3. Tier 1 includes students and assistants, specifically PhD students, Postgraduate Teaching Assistants, Research Assistants, and Teaching Assistants. Tier 0 represents non-academic or ambiguous titles which are filtered from the dataset.

The filtering approach removes profiles with unclear institutional standing rather than forcing classification of ambiguous cases. This conservative strategy prioritises data quality over sample size, as misclassification would introduce more noise than exclusion. The filtered group consists predominantly of "Unknown" titles (83 of 105 profiles, 79.05%), with the remainder comprising administrative positions, technical support roles, and vague designations such as "Researcher" or "Affiliate Academic" that lack clear rank indicators. In total, 105 profiles (17.26% of the original dataset) are filtered, leaving 503 profiles for analysis.

Honorary titles such as Honorary Professor or Honorary Lecturer are retained at their corresponding tier levels. These designations reflect formal academic recognition by UCL and indicate genuine institutional standing across all career stages. The dataset contains seven honorary appointments that would otherwise be classified as Tier 0, demonstrating that honorary status spans multiple seniority levels.

The seniority hierarchy is encoded as a single ordinal numeric variable rather than binary indicator flags. This design choice reflects the inherently ordered nature of academic ranks, where the relationship 1 < 2 < 3 < 4 carries meaningful interpretation. The continuous encoding simplifies subsequent standardisation procedures and avoids the redundancy of four separate binary features. Whilst numeric encoding assumes approximately equal intervals between tiers, this representation allows clustering algorithms to incorporate hierarchical distance when evaluating similarity between profiles.

A known limitation of the classification logic is its reliance on exact pattern matching for chair designations. Titles containing typographical errors or non-standard formatting (such as "DeepMind Chair in Artifical Intelligence" with a misspelled "Artificial") are not captured by the "Chair of" or "Chair in" patterns and are consequently filtered. Implementing fuzzy matching or error correction would increase code complexity without substantially improving coverage, given the rarity of such cases in the dataset.

In [8]:
df = df.with_columns([
    (pl.col('latest_publication_year') > 0).cast(pl.Int8).alias('has_publications')
])

df = df.with_columns([
    pl.col('latest_publication_year').clip(0, 2026).alias('latest_publication_year')
])

df = df.with_columns(
    pl.when(pl.col('has_publications') == 0)
    .then(0)
    .otherwise(
        (pl.col('latest_publication_year') - pl.col('first_publication_year')).clip(1, None)
    )
    .alias('career_years')
)

df = df.with_columns([
    (pl.col('h_index') / pl.col('career_years')).fill_nan(0).alias('m_quotient'),
    (pl.col('publication_count') / pl.col('career_years')).fill_nan(0).alias('productivity_rate'),
    (pl.col('total_citations') / pl.col('career_years')).fill_nan(0).alias('annual_impact'),
    (pl.col('recent_publications') / pl.col('publication_count')).fill_nan(0).alias('recent_activity_ratio')
])

df = df.with_columns([
    (pl.col('solo_publications') / pl.col('publication_count')).fill_nan(0).alias('solo_work_rate'),
    (pl.col('journal_article_count') / pl.col('publication_count')).fill_nan(0).alias('journal_article_rate'),
    (pl.col('preprint_count') / pl.col('publication_count')).fill_nan(0).alias('preprint_rate'),
])

df = df.with_columns([
    (pl.col('teaching_module_count') > 0).cast(pl.Int8).alias('has_teaching_experience')
])

df = df.with_columns([
    pl.when(pl.col('latest_publication_year') == 0)
    .then(0)
    .otherwise(2026 - pl.col('latest_publication_year'))
    .alias('years_since_last_pub')
])

df = df.drop(['first_publication_year', 'latest_publication_year', 'teaching_modules'])

print(f"After career features: {df.shape}")

After career features: (503, 48)


## Section 5: Career and Publication Features

Research productivity and impact vary significantly by career stage and disciplinary norms. Normalising raw bibliometric measures by career length and decomposing publication patterns enables fair comparison across academics at different career stages whilst revealing individual collaboration styles, publication preferences, and temporal research activity.

Data quality issues identified during exploratory analysis required correction before feature engineering. One profile contained a future publication date of 2038, likely a data entry error, whilst 128 profiles (25.45% of the filtered dataset) had `latest_publication_year` set to zero, indicating no recorded publications. Future publication dates are capped at 2026 using a clip operation to eliminate the erroneous entry whilst preserving zero values, which represent legitimate non-publishing academics such as students (60.16% of non-publishers), teaching-focused staff, and recent hires.

A binary publication status indicator `has_publications` distinguishes academics who have published from those without recorded publications. This flag provides essential context for interpreting subsequent temporal features, particularly `years_since_last_pub`, where a value of zero can indicate either "no publications" or "published in the current year" depending on publication status. The 128 non-publishers represent valid cases distributed across all seniority tiers, with the majority being Tier 1 students (77 profiles) but also including established academics in Tiers 2-4.

Career span is calculated as the difference between latest and first publication years, representing an approximation of active research duration. For profiles with publications, career span is clipped to a minimum of one year to handle cases where all publications occur in a single year, preventing division by zero in subsequent normalised metrics. Non-publishers are assigned a career span of zero to avoid inflating normalised productivity metrics with artificial denominators. This approach ensures that division operations on non-publishers yield zero rather than undefined values.

Four academic impact metrics are derived by normalising bibliometric measures by career length. The m-quotient, calculated as h-index divided by career years, represents the rate of h-index accumulation and serves as a career-normalised measure of sustained research impact (Hirsch, 2005, p. 16569). Values exceeding 1.0 indicate above-average impact relative to career length. The productivity rate, calculated as total publications divided by career years, measures annual publication output as an indicator of research quantity. Annual impact, calculated as total citations divided by career years, captures the rate of citation accumulation as a measure of research visibility and influence. The recent activity ratio, calculated as recent publications (within the past two years) divided by total publication count, indicates the proportion of recent work in an academic's portfolio. High values suggest currently active researchers, whilst low values indicate primarily historical productivity. All ratio calculations include `.fill_nan(0)` operations to handle division by zero for non-publishers explicitly.

Three publication type composition features capture disciplinary publication cultures and collaboration patterns. The solo work rate, calculated as solo publications divided by total publications, ranges from 0.0 for fully collaborative researchers to 1.0 for those publishing exclusively alone. The journal article rate, calculated as journal articles divided by total publications, reflects preference for journal versus conference publication venues, which varies substantially across computer science subfields. The preprint rate, calculated as preprints divided to total publications, serves as an indicator of engagement with open science practices and rapid dissemination channels. Missing values arising from zero publication counts are filled with zeros using `.fill_nan(0)`.

Teaching engagement is captured through a binary indicator `has_teaching_experience`, which flags academics with at least one recorded teaching module. A binary approach is preferred over quantitative metrics because the source variable `teaching_module_count` measures module diversity rather than teaching load. A single year-round module represents substantially different effort than ten small seminars, making raw counts misleading for measuring teaching commitment. The binary flag simply distinguishes teaching-involved academics from those without recorded teaching responsibilities.

Two temporal features characterise publication recency. The publication status flag `has_publications`, described earlier, provides binary classification. The feature `years_since_last_pub` calculates the time elapsed since the most recent publication, computed as 2026 minus the latest publication year. For non-publishers, this value is set to zero, with disambiguation provided by the `has_publications` flag. Among the 375 publishers (74.55% of the dataset), the distribution shows 82.40% published within the past two years (active researchers), 10.13% have a three-to-five-year gap (moderate inactivity), and 7.47% have not published in six or more years (inactive researchers). This two-feature design separates the concepts of "never published" from "published X years ago" without introducing arbitrary placeholder values.

Following feature creation, the source columns `first_publication_year`, `latest_publication_year`, and `teaching_modules` are removed from the dataset. These fields have been transformed into derived features that better capture their analytical content. The text field `teaching_modules` cannot be meaningfully analysed without natural language processing techniques beyond the scope of this analysis, and the binary teaching flag suffices for clustering purposes.

Exploratory analysis revealed substantial right skewness in multiple features, including career years (skewness = 1.28), years since last publication (skewness = 3.01), and various publication count metrics (skewness ranging from 2.05 to 10.39). This skewness reflects the typical distribution of academic productivity, where a small number of highly productive researchers produce disproportionate output. These distributional properties are preserved in the raw features and will be addressed through transformation in subsequent preprocessing stages to meet the assumptions of distance-based clustering algorithms.

In [9]:
# Separate identity columns from features for post-clustering interpretation
# Identity mapping enables linking cluster assignments back to individual profiles
identity_df = df.select(['_index', 'name', 'email', 'profile_id'])
identity_df.write_parquet('../data/processed/ucl_identity_mapping.parquet')
print(f"Identity mapping saved: {identity_df.shape}")

df = df.drop(['name', 'email', 'profile_id'])

# Save unscaled features for human-readable interpretation before transformations
df.write_parquet('../data/processed/ucl_features_unscaled.parquet')
print(f"Unscaled features saved: {df.shape}")

Identity mapping saved: (503, 4)
Unscaled features saved: (503, 45)


Identity columns (name, email, profile_id) are separated from the feature matrix and saved to a dedicated mapping file to enable post-clustering interpretation whilst removing non-numerical data from the analysis pipeline. The _index column serves as the linking key between the feature matrix and identity mapping throughout subsequent transformations.

In [10]:
features_to_log = [
    'publication_count',
    'journal_article_count',
    'preprint_count',
    'recent_publications',
    'total_citations',
    'avg_citations_per_publication',
    'max_citations',
    'h_index',
    'i10_index',
    'avg_coauthors',
    'max_coauthors',
    'solo_publications',
    'has_doi_count',
    'teaching_module_count',
    'm_quotient',
    'productivity_rate',
    'annual_impact',
    'career_years',
    'years_since_last_pub',
]

features_to_keep_raw = [
    '_index',
    'seniority_level',
    'has_research_interests',
    'has_bio',
    'has_teaching_info',
    'has_website',
    'is_ml_ai_focus',
    'is_vision_focus',
    'is_nlp_focus',
    'is_security_focus',
    'is_systems_focus',
    'is_theory_focus',
    'is_hci_focus',
    'is_robotics_focus',
    'is_data_science_focus',
    'is_graphics_focus',
    'is_bioinformatics_focus',
    'is_quantum_focus',
    'is_networks_focus',
    'is_software_eng_focus',
    'has_teaching_experience',
    'has_publications',
    'recent_activity_ratio',
    'solo_work_rate',
    'journal_article_rate',
    'preprint_rate',
]

df_transformed = df.select([
    *[(pl.col(col) + 1).log().alias(f'log_{col}') for col in features_to_log],
    *[pl.col(col) for col in features_to_keep_raw]
])

print(f"After transformation: {df_transformed.shape}")
print(f"  Log-transformed: {len([c for c in df_transformed.columns if c.startswith('log_')])}")
print(f"  Raw features: {len([c for c in df_transformed.columns if not c.startswith('log_') and c != '_index'])}")
print(f"  Index column: 1")

After transformation: (503, 45)
  Log-transformed: 19
  Raw features: 25
  Index column: 1


## Section 6: Log Transformation

Skewed distributions reduce clustering performance by assigning disproportionate weight to extreme values in distance calculations. Log transformation addresses this by compressing the range of large values whilst preserving rank ordering and reducing the influence of outliers on similarity measures.

The transformation strategy applies log(x+1) to features exhibiting high skewness whilst preserving features with symmetric distributions or those where transformation would be counterproductive. The choice of log(x+1) rather than log(x) handles zero values, which are common in publication metrics, as log(0) is undefined. Adding one before transformation ensures all values remain defined and positive.

Feature selection for transformation follows the statistical convention that features with absolute skewness exceeding 1.0 are considered highly skewed and warrant transformation (Bulmer, 1979, p. 63). This threshold provides an objective criterion for identifying distributions where extreme values dominate and distance-based clustering algorithms would struggle with untransformed data. Skewness values for all features were validated during exploratory data analysis.

Nineteen features receive log transformation, comprising three categories of highly skewed variables. Publication and citation metrics including publication counts, citation totals, h-index, i10-index, and collaboration measures (average and maximum coauthors) exhibit substantial right skewness due to the concentration of academic productivity among a subset of highly active researchers. Career-normalised metrics derived from these features (m-quotient, productivity rate, annual impact) inherit the skewness properties of their source variables despite normalisation. Temporal features (career years and years since last publication) similarly display right-skewed distributions reflecting the presence of both early-career and established academics in the dataset.

Twenty-five features are retained in their raw form across four categories. Binary indicators, including profile completeness flags (has_research_interests, has_bio, has_teaching_info, has_website), research domain classifications (14 domain flags), and engagement indicators (has_teaching_experience, has_publications), require no transformation as they encode discrete presence or absence. The ordinal feature seniority_level represents an inherently ranked categorical variable (1-4 scale) where transformation would destroy the meaningful hierarchical structure.

Ratio features (recent_activity_ratio, solo_work_rate, journal_article_rate, preprint_rate) are retained raw despite some exhibiting high skewness (solo_work_rate: 3.27, preprint_rate: 3.22). These features are bounded [0,1], which naturally constrains extreme values and prevents the outlier dominance that justifies transformation in unbounded features. Log transformation of bounded data introduces numerical instability near zero, as log(0.01) produces large negative values that are difficult to interpret and can distort distance calculations. The structural bounds of ratio features ensure all values remain within a reasonable range for clustering algorithms regardless of their distributional shape.

The row identifier _index is preserved without transformation to maintain linkage to the identity mapping file throughout subsequent preprocessing stages.

In [11]:
log_features = [c for c in df_transformed.columns if c.startswith('log_')]
raw_features = [c for c in df_transformed.columns if not c.startswith('log_') and c != '_index']

X_log = df_transformed.select(log_features).to_numpy()
X_raw = df_transformed.select(raw_features).to_numpy()
index_col = df_transformed.select('_index').to_numpy()

scaler = StandardScaler()
X_log_scaled = scaler.fit_transform(X_log)

X_features = np.concatenate([X_log_scaled, X_raw, index_col], axis=1)

feature_columns = [f"{col}_scaled" for col in log_features] + raw_features + ['_index']

df_features = pl.DataFrame(X_features, schema=feature_columns)

print(f"After scaling: {df_features.shape}")

After scaling: (503, 45)


## Section 7: Feature Scaling

Standardisation ensures features contribute equally to distance calculations in clustering algorithms by placing all variables on comparable scales. However, different feature types require different scaling strategies to preserve their inherent properties and interpretability.

The nineteen log-transformed features undergo z-score normalisation using StandardScaler, which transforms each feature to have mean zero and standard deviation one. This scaling is necessary because log-transformed metrics span different numerical ranges despite transformation. For example, log-transformed publication counts and log-transformed citation counts operate on different scales, and without standardisation, features with larger variance would dominate distance calculations in clustering algorithms.

The twenty-five raw features are retained without scaling to preserve their semantic meaning and natural interpretability. Binary flags (profile completeness indicators, research domain classifications, and engagement flags) already operate on a comparable 0-1 scale and represent discrete categorical information where standardisation would be meaningless. Ratio features (recent_activity_ratio, solo_work_rate, journal_article_rate, preprint_rate) are already normalised to the [0,1] interval through their construction as proportions, eliminating the need for further scaling. The ordinal feature seniority_level represents a ranked hierarchy (1-4 scale) where the natural spacing between levels carries interpretable meaning that would be distorted by z-score transformation.

The row identifier _index is preserved without scaling as it serves solely as an integer key for linking clustered observations back to the identity mapping file.

Following scaling, log-transformed features are concatenated with raw features and the index column to create the final feature matrix. This selective scaling approach balances the need for equal feature contribution in distance-based clustering with the preservation of meaningful structure in categorical and bounded numerical features.

In [12]:
# Save final feature matrix for clustering
df_features.write_parquet('../data/processed/ucl_features_final.parquet')
print(f"Final features saved: {df_features.shape}")

Final features saved: (503, 45)
